### Environment Setup & Helper Function

First, we set up the connection to the local Ollama server to use a free local LLM.

In [1]:
# 1. Install Dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,721 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/ma

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# 2. Start Ollama Server in Background
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10) # Give the server time to spin up
print("Ollama server ready on http://localhost:11434")

Ollama server ready on http://localhost:11434


In [4]:
# 3. Pull Llama 3 Model (Free Local LLM)
!ollama pull llama3

In [5]:
# 4. Standard API Helper with Parameter Control
import requests
import json

OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, model="llama3", temperature=0.7, stream=False):
    """Helper function to interact with the local Ollama API."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": {
            "temperature": temperature
        }
    }

    resp = requests.post(OLLAMA_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Request failed: {resp.status_code}, {resp.text}")

    if stream:
        for line in resp.text.splitlines():
            if not line.strip(): continue
            obj = json.loads(line)
            print(obj.get("response", ""), end="")
        print("\n")
    else:
        return resp.json().get("response", "")

print("Environment Ready. Target API:", OLLAMA_API_URL)

Environment Ready. Target API: http://localhost:11434/api/generate


### Case Study Context: GreenLeaf Co.

**The Scenario:** GreenLeaf Co. is a small organic wellness brand. Their marketing team of 2 people needs to produce 50+ pieces of content per month (product descriptions, social media posts, email subjects, ad copy) while maintaining a consistent brand voice. We will use Prompt Engineering and our local LLM to build a scalable content generation pipeline.

### Topic 1: Zero-Shot Prompting

**Concept:** Asking the model to perform a task without providing any prior examples.

**Practical Scenario:** The team's very first attempt to generate a product description for their new organic green tea. Let's see what happens with a simple request.

In [6]:
# The model relies entirely on its pre-trained weights, with zero brand context.
zero_shot_prompt = """
Write a product description for our organic green tea.
"""

response = call_ollama(zero_shot_prompt, model="llama3", temperature=0.7)
print(f"Prompt:\n{zero_shot_prompt.strip()}")
print(f"\nResult:\n{response.strip()}\n")

Prompt:
Write a product description for our organic green tea.

Result:
**Introducing Our Pure and Refreshing Organic Green Tea**

Indulge in the tranquility of nature with every sip of our USDA-certified organic green tea. Sourced from the finest, non-GMO farms, our tea leaves are carefully hand-picked to ensure a smooth and refreshing brew.

Our organic green tea is a symphony of subtle flavors, with notes of light grassy hints and a hint of sweetness. The aroma is uplifting and calming, perfect for unwinding after a long day or as a revitalizing pick-me-up.

**Why Choose Our Organic Green Tea?**

* **100% USDA-certified organic**: We're committed to sustainable farming practices that protect the environment and promote biodiversity.
* **Non-GMO**: You can trust our tea is free from genetically modified organisms (GMOs).
* **High-quality leaves**: Our carefully selected tea leaves ensure a consistent flavor profile and aroma.
* **No artificial additives**: Our tea is free from artifi

**Analysis:** As seen in the slides, a zero-shot output is typically too generic. It lacks GreenLeaf's specific sensory brand voice, misses key benefits, and forgets to include a Call to Action (CTA). The AI needs more structure.

### Topic 2: Few-Shot Prompting

**Concept:** Providing the model with a few examples (input-output pairs) to establish a clear pattern, tone, and format before asking it to process the actual target data.

**Practical Scenario:** We will teach the LLM GreenLeaf's specific brand voice (sensory language, ~60 words, clear CTA) by showing it two past successful product descriptions.

In [7]:
few_shot_prompt = """
Write a product description in our brand style.

Product: Lavender Sleep Oil
Description: "Drift into deep, restorative sleep with our pure lavender essential oil. Sourced from Provençal fields, this calming blend relaxes your mind and body. Just 3 drops. Shop our wellness collection."

Product: Turmeric Supplement
Description: "Fight inflammation naturally with our potent turmeric capsules. 500mg curcumin, 95% bioavailability. Your daily dose of ancient healing. Order now."

Product: Organic Green Tea
Description:
"""

response = call_ollama(few_shot_prompt, model="llama3", temperature=0.3)
print("Few-Shot Result:")
print(response.strip())

Few-Shot Result:
Here are the product descriptions in your brand style:

**Lavender Sleep Oil**
"Indulge in a peaceful slumber with our soothing Lavender Sleep Oil. Sourced from the picturesque fields of Provence, this calming blend whispers sweet dreams to your mind and body. With just 3 drops, you'll be drifting off into a restful night's sleep in no time. Shop our wellness collection for more natural solutions."

**Turmeric Supplement**
"Unlock the ancient power of turmeric with our potent capsules. Each serving delivers 500mg of curcumin, ensuring maximum bioavailability (95%) to tackle inflammation and promote overall well-being. Your daily dose of natural healing is just a click away. Order now and start feeling the benefits."

**Organic Green Tea**
"Start your day off right with a cup of our premium Organic Green Tea. Sourced from sustainable farms, this refreshing brew is packed with antioxidants and subtle flavor. Whether you're looking to boost your energy or simply savor the

### Topic 3: Role Prompting

**Concept:** Assigning a specific persona, profession, or identity to the LLM to drastically alter its vocabulary, tone, and perspective.

**Practical Scenario:** The team needs an Instagram post for the tea. Social media requires a very different voice from formal product pages. We unlock this by assigning a specialized role.

In [8]:
# Persona: Gen-Z Social Media Strategist
role_prompt = """
You are a Gen Z social media strategist for a wellness brand.
You specialize in Instagram content that drives saves and shares. Use emojis sparingly but effectively.

Task: Write an Instagram caption for our new Organic Green Tea.
"""

print("Role Prompt Output (Instagram):")
print(call_ollama(role_prompt, model="llama3", temperature=0.8))

Role Prompt Output (Instagram):
"Steep yourself in calm 🌿 Introducing our newest obsession: Organic Green Tea! 💚 This refreshing brew is more than just a drink - it's a moment of mindfulness in your busy day. With its subtle hint of grassy goodness and zero artificial anything, you'll be sipping like a zen master in no time ☕️ Treat yourself to the ultimate self-care ritual: take a deep breath, pour a cup, and let the stress melt away. Try it out and tag us in your calmest moments! #OrganicGreenTea #WellnessWednesday"


### Topic 4: Prompt Templates

**Concept:** Using Python string formatting to dynamically generate prompts based on reusable structures and placeholder variables (`{{variable}}`).

**Practical Scenario:** GreenLeaf wants a master template they can use for ANY product and ANY content type without writing a new prompt from scratch every time.

In [9]:
def generate_greenleaf_content(role, domain, audience, tone, format_type, content_type, topic, key_points, word_count, cta):
    """A reusable master prompt template function."""

    template = f"""
    You are a {role} specializing in {domain}. Your audience is {audience}.
    Tone: {tone}. Format: {format_type}.

    Task: Write a {content_type} about {topic}.
    Key points to include: {key_points}
    Length: Around {word_count} words.
    CTA: {cta}
    """

    return call_ollama(template, model="llama3", temperature=0.7)

# Reusing the master template for a Promotional Email Campaign
email_content = generate_greenleaf_content(
    role="expert copywriter",
    domain="e-commerce",
    audience="health-conscious shoppers",
    tone="warm & inspiring",
    format_type="2 short paragraphs",
    content_type="promotional email",
    topic="Organic Green Tea",
    key_points="antioxidants, farm-sourced, relaxation",
    word_count=100,
    cta="Shop the Harvest"
)

print("Template Output (Email Campaign):\n")
print(email_content.strip())

Template Output (Email Campaign):

Dear friend,

As you navigate the hustle and bustle of everyday life, I want to invite you to take a moment for yourself. Our Organic Green Tea is more than just a refreshing beverage - it's an opportunity to slow down, relax, and nourish your body and mind. Sourced from the finest organic farms, our tea is packed with antioxidants that work tirelessly to protect your cells from damage.

When you sip on this delicious brew, you're not just quenching your thirst - you're giving your body a boost of vitality and calm. So why not take a moment to unwind and indulge in some much-needed self-care? Shop the Harvest today and discover the soothing benefits of our Organic Green Tea for yourself!


### Topic 5: Full Content Generation Pipeline

**Concept:** Combining Prompt Templates, Role Prompting, and structured formatting instructions to build an automated end-to-end pipeline.

**Practical Scenario:** Generating the entire marketing suite (Product Description, Instagram Caption, and Google Ad Copy) for a new wellness product in a single run.

In [10]:
def run_greenleaf_pipeline(product_name, key_benefits):
    """Generates a multi-channel marketing campaign in one pass."""

    pipeline_prompt = f"""
    You are the Chief Marketing Officer for GreenLeaf Co., an organic wellness brand.
    We are launching a new product.

    Product Context:
    - Product Name: {product_name}
    - Key Benefits: {key_benefits}

    Task:
    Generate a cohesive marketing campaign consisting of three parts.
    Format your response strictly using Markdown headers for each section.

    # 1. Product Description
    Tone: Sensory and informative. Length: exactly 60 words. Must end with a Call to Action.

    # 2. Instagram Caption
    Tone: Casual, Gen-Z strategist. Drive saves and shares. Use a maximum of 3 emojis and exactly 3 relevant hashtags.

    # 3. Google Search Ad
    Write a Google Ad headline (max 30 characters) and description (max 90 characters). Tone: Urgent and conversion-focused.
    """

    # Using stream=True for a better UX during live demonstrations
    print(f" Launching Campaign Pipeline for: {product_name}...\n")
    call_ollama(pipeline_prompt, model="llama3", temperature=0.7, stream=True)

# Run the final pipeline on a brand new product
run_greenleaf_pipeline(
    product_name="Himalayan Pink Salt Body Scrub",
    key_benefits="Exfoliates dead skin, mineral-rich, leaves skin glowing and hydrated, 100% natural"
)

 Launching Campaign Pipeline for: Himalayan Pink Salt Body Scrub...

Here is the marketing campaign for GreenLeaf Co.'s Himalayan Pink Salt Body Scrub:

# 1. Product Description
**Glow Up with Nature's Best**
Get ready to reveal radiant skin with our Himalayan Pink Salt Body Scrub! This 100% natural wonder exfoliates dead skin, infuses minerals, and leaves your skin hydrated and glowing. Try it now and unlock a healthier, more confident you! [Shop Now](https://www.greenleafco.com/himalayan-pink-salt-body-scrub)

# 2. Instagram Caption
"Ready to level up your self-care game? üå∏ Our Himalayan Pink Salt Body Scrub is like a spa day in a jar! üíÜ‚Äç‚ôÄÔ∏è Exfoliate, hydrate, and glow with this all-natural goodness. #organicbeauty #glowup"

# 3. Google Search Ad
**Headline:** Reveal Radiant Skin Now!
**Description:** Get the best of nature's exfoliating power! Himalayan Pink Salt Body Scrub by GreenLeaf Co. - 100% natural, hydrating, and glowing skin guaranteed! Try now and see the diffe